# Marketing attribution

Generated from the FeatureMesh docs tutorial. This variant targets the FeatureMesh demos Jupyter environment.


## Set up FeatureMesh

Load the Jupyter magic and create a local `BatchClient` for the demos Jupyter environment. Run these cells once before the tutorial.


In [1]:
%load_ext featuremesh

In [2]:
from featuremesh import BatchClient, set_default

client = BatchClient()
set_default("client", client)
print("FeatureMesh BatchClient ready (local DuckDB)")

FeatureMesh BatchClient ready (local DuckDB)


This tutorial allocates the same purchase revenue five different ways so you can see exactly what an attribution model changes.

You will compare first touch, last touch, linear, time-decay, and channel-weighted credit. You will also shorten the lookback window and watch previously eligible touches disappear.

It uses the entity bindings from [E-commerce](https://featuremesh.com/docs/tutorials/analytics/ecomm), including a `NESTED` binding for a child grain. The new working pattern is `TRANSFORM()`: read each nested `RETURN` as a small SQL query over one conversion's array of touches. In the notebook, run the Data and Model sections first.


## Meet the two journeys

The whole dataset is two purchases and six touchpoints:


In [3]:
%%featureql --client client --hide-dataframe

DROP FEATURES IF EXISTS IN FM.MKT UP TO LEVEL 9;

In [4]:
%%featureql --client client

CREATE SCHEMA IF NOT EXISTS tutorial_mkt;
--
DROP TABLE IF EXISTS tutorial_mkt.conversions;
--
DROP TABLE IF EXISTS tutorial_mkt.touchpoints;
--
CREATE TABLE tutorial_mkt.touchpoints (
  id BIGINT,
  user_id BIGINT,
  campaign VARCHAR,
  channel VARCHAR,
  touchpoint_type VARCHAR,
  ts TIMESTAMP
);
--
INSERT INTO tutorial_mkt.touchpoints VALUES
  (1, 1, 'Blog', 'organic', 'blog_visit', TIMESTAMP '2024-01-05 10:00:00'),
  (2, 1, 'Search', 'google', 'ad_click', TIMESTAMP '2024-01-12 14:00:00'),
  (3, 1, 'Display', 'meta', 'ad_impression', TIMESTAMP '2024-01-15 10:00:00'),
  (4, 1, 'Email', 'email', 'email_click', TIMESTAMP '2024-01-18 11:00:00'),
  (5, 2, 'Search', 'google', 'ad_click', TIMESTAMP '2024-01-08 09:00:00'),
  (6, 2, 'Email', 'email', 'email_click', TIMESTAMP '2024-01-14 10:00:00');
--
CREATE TABLE tutorial_mkt.conversions (
  id BIGINT,
  user_id BIGINT,
  revenue BIGINT,
  conv_ts TIMESTAMP
);
--
INSERT INTO tutorial_mkt.conversions VALUES
  (1, 1, 90, TIMESTAMP '2024-01-20 15:00:00'),
  (2, 2, 60, TIMESTAMP '2024-01-15 16:00:00');
--
SELECT CAST(COUNT(*) AS INTEGER) AS cnt FROM tutorial_mkt.conversions;

,cnt
0,2


- **User 1**, $90 on January 20: Blog → Search → Display impression → Email
- **User 2**, $60 on January 15: Search → Email

User 1's Display touch is an impression rather than a click. That gives the weighted model a reason to assign it less credit.

## Model conversions and touchpoints

Conversions define the outer grain: each calculation begins with one purchase and its revenue. Touchpoints belong to a user. `LOOKBACK_DAYS` is an input, so the eligibility window can change without rewriting the attribution formula.


In [5]:
%%featureql --client client

CREATE OR REPLACE FEATURES IN FM.MKT AS
PERSIST
    conversions := ENTITY(),
    users := ENTITY(),
    conversion_id := INPUT(BIGINT#conversions),
    user_id := INPUT(BIGINT#users),
    lookback_days := INPUT(BIGINT)
;

,feature_name,status,message
0,FM.MKT.CONVERSIONS,CREATED,Feature created as not exists
1,FM.MKT.USERS,CREATED,Feature created as not exists
2,FM.MKT.CONVERSION_ID,CREATED,Feature created as not exists
3,FM.MKT.USER_ID,CREATED,Feature created as not exists
4,FM.MKT.LOOKBACK_DAYS,CREATED,Feature created as not exists


Map the two tables. A conversion points to its user with `user_id`. The touchpoints table is bound to `USER_ID`, so each user's touches can be gathered:


In [6]:
%%featureql --client client

CREATE OR REPLACE FEATURES IN FM.MKT AS
PERSIST
    tables.conversions := EXTERNAL_COLUMNS(
        id BIGINT#conversions BIND TO conversion_id,
        user_id BIGINT#users,
        revenue BIGINT,
        conv_ts TIMESTAMP
        FROM TABLE(tutorial_mkt.conversions)
    ),
    tables.touchpoints := EXTERNAL_COLUMNS(
        id BIGINT,
        user_id BIGINT#users BIND TO user_id,
        campaign VARCHAR,
        channel VARCHAR,
        touchpoint_type VARCHAR,
        ts TIMESTAMP
        FROM TABLE(tutorial_mkt.touchpoints)
    )
;

,feature_name,status,message
0,FM.MKT.TABLES.CONVERSIONS,CREATED,Feature created as not exists
1,FM.MKT.TABLES.TOUCHPOINTS,CREATED,Feature created as not exists


Every model starts with the same qualifying set. A touch counts only when it:

1. belongs to the converting user;
2. occurred before the purchase; and
3. falls within the bound lookback window.

Persist that set once. `USER_TOUCHES` gathers the touches of the conversion's user with `RELATED()`. `QUALIFYING_TOUCHES` keeps the ones inside the window, oldest first. `CARRY()` brings the conversion time and the lookback into the array, where the filter runs on each touch:


In [7]:
%%featureql --client client

CREATE OR REPLACE FEATURES IN FM.MKT AS
PERSIST
    conversion_user_id := tables.conversions[user_id],
    conversion_revenue := tables.conversions[revenue],
    conversion_ts := tables.conversions[conv_ts],
    user_touches := conversion_user_id.RELATED(
        ARRAY_AGG(
            ROW(
                tables.touchpoints[campaign] AS campaign,
                tables.touchpoints[channel] AS channel,
                tables.touchpoints[touchpoint_type] AS touchpoint_type,
                tables.touchpoints[ts] AS ts
            )
        )
        GROUP BY (tables.touchpoints[user_id])
    ),
    qualifying_touches := user_touches.CARRY(conversion_ts AS conversion_ts).CARRY(
        lookback_days AS lookback_days
    ).TRANSFORM(
        RETURN
            campaign,
            channel,
            touchpoint_type,
            ts
        WHERE ts < conversion_ts
        AND ts >= DATE_ADD(conversion_ts, -lookback_days, 'day')
        ORDER BY ts
    )
;

,feature_name,status,message
0,FM.MKT.CONVERSION_USER_ID,CREATED,Feature created as not exists
1,FM.MKT.CONVERSION_REVENUE,CREATED,Feature created as not exists
2,FM.MKT.CONVERSION_TS,CREATED,Feature created as not exists
3,FM.MKT.USER_TOUCHES,CREATED,Feature created as not exists
4,FM.MKT.QUALIFYING_TOUCHES,CREATED,Feature created as not exists


Every query below binds the two conversions and their two users (`NESTED USER_ID`, the child grain that `RELATED()` reads), with a 30-day lookback unless stated:


In [8]:
%%featureql --client client

RETURN
    conversion_id,
    conversion_revenue,
    campaigns := qualifying_touches[campaign]
FROM FM.MKT
FOR
    conversion_id := BIND_VALUES(ARRAY(1, 2)),
    NESTED user_id := BIND_VALUES(ARRAY(1, 2)),
    lookback_days := BIND_VALUE(30)
ORDER BY conversion_id
;

,FM.MKT.CONVERSION_ID,FM.MKT.CONVERSION_REVENUE,CAMPAIGNS
0,1,90,"[Blog, Search, Display, Email]"
1,2,60,"[Search, Email]"


The sections that follow change how credit is divided, not which touches qualify.

## Give all credit to the first touch

Assign each purchase entirely to its earliest qualifying touch:


In [9]:
%%featureql --client client

DEFINE
    first_campaign := qualifying_touches[campaign][1]
RETURN
    campaign := first_campaign,
    attributed := SUM(conversion_revenue) GROUP BY (first_campaign)
FROM FM.MKT
FOR
    conversion_id := BIND_VALUES(ARRAY(1, 2)),
    NESTED user_id := BIND_VALUES(ARRAY(1, 2)),
    lookback_days := BIND_VALUE(30)
ORDER BY campaign
;

,CAMPAIGN,ATTRIBUTED
0,Blog,90
1,Search,60


User 1's **90** goes to Blog. User 2's **60** goes to Search.

## Give all credit to the last touch

Keep the same journeys and window, but select the latest touch instead:


In [10]:
%%featureql --client client

DEFINE
    last_campaign := ELEMENT_AT_POS(qualifying_touches[campaign], -1)
RETURN
    campaign := last_campaign,
    attributed := SUM(conversion_revenue) GROUP BY (last_campaign)
FROM FM.MKT
FOR
    conversion_id := BIND_VALUES(ARRAY(1, 2)),
    NESTED user_id := BIND_VALUES(ARRAY(1, 2)),
    lookback_days := BIND_VALUE(30)
ORDER BY campaign
;

,CAMPAIGN,ATTRIBUTED
0,Email,150


Both journeys end on Email, so Email receives the full **150**.

The underlying events did not change. Only the rule for choosing a touch changed.

## Split credit evenly

Linear attribution divides each conversion equally across all of its qualifying touches. `COUNT() OVER ()` counts the touches of one conversion, and `UNNEST()` turns the credits of every conversion into rows that `GROUP BY` can add up:


In [11]:
%%featureql --client client

DEFINE
    credits := qualifying_touches.CARRY(conversion_revenue AS revenue).TRANSFORM(
        RETURN
            campaign,
            credit := revenue::DOUBLE / COUNT() OVER ()
    ),
    credit := UNNEST(credits)
RETURN
    campaign := credit[campaign],
    attributed := UNSAFE SUM(credit[credit]) GROUP BY (credit[campaign])
FROM FM.MKT
FOR
    conversion_id := BIND_VALUES(ARRAY(1, 2)),
    NESTED user_id := BIND_VALUES(ARRAY(1, 2)),
    lookback_days := BIND_VALUE(30)
ORDER BY campaign
;

,CAMPAIGN,ATTRIBUTED
0,Blog,22.5
1,Display,22.5
2,Email,52.5
3,Search,52.5


User 1 has four touches, so each receives **22.5**. User 2 has two, so each receives **30**. After adding the journeys together, Search and Email each receive **52.5**; Blog and Display each receive **22.5**. Each conversion still allocates exactly 100% of its revenue.

A credit is a share of revenue, a ratio, so the compiler marks it not additive and refuses a plain `SUM()`. Here summing is the intent: the credits of one conversion add back to its revenue. `UNSAFE SUM()` states that override explicitly, and the result carries the mark.

## Shorten the lookback window

Return to first-touch attribution but bind `LOOKBACK_DAYS` to **7** instead of 30:


In [12]:
%%featureql --client client

DEFINE
    first_campaign := qualifying_touches[campaign][1]
RETURN
    campaign := first_campaign,
    attributed := SUM(conversion_revenue) GROUP BY (first_campaign)
FROM FM.MKT
FOR
    conversion_id := BIND_VALUES(ARRAY(1, 2)),
    NESTED user_id := BIND_VALUES(ARRAY(1, 2)),
    lookback_days := BIND_VALUE(7)
ORDER BY campaign
;

,CAMPAIGN,ATTRIBUTED
0,Display,90
1,Email,60


User 1's Blog and Search touches are now too old, so the first eligible touch becomes Display and receives **90**. User 2's Search touch also drops out, moving **60** to Email.

This result shows why the lookback is part of the model, not a reporting footnote: it changes which channels can receive credit.

## Favor recent touches with time decay

Time-decay attribution avoids choosing only one touch. Each touch gets a weight of `0.5^(days_before_conversion / 7)`, which halves every 7 days. Weights are normalized to one within each journey, then revenue is allocated:


In [13]:
%%featureql --client client

DEFINE
    credits := qualifying_touches.CARRY(conversion_ts AS conversion_ts).CARRY(
        conversion_revenue AS revenue
    ).TRANSFORM(
        DEFINE
            weight := POW(0.5e0, DATE_SUBTRACT(conversion_ts, ts, 'day')::DOUBLE / 7e0)
        RETURN
            campaign,
            credit := revenue::DOUBLE * weight / SUM(weight) OVER ()
    ),
    credit := UNNEST(credits)
RETURN
    campaign := credit[campaign],
    attributed := UNSAFE SUM(credit[credit]) GROUP BY (credit[campaign])
FROM FM.MKT
FOR
    conversion_id := BIND_VALUES(ARRAY(1, 2)),
    NESTED user_id := BIND_VALUES(ARRAY(1, 2)),
    lookback_days := BIND_VALUE(30)
ORDER BY campaign
;

*** Warning(s) (1) ***

  1. [SUMMARIZABILITY-CONTRACT-UNKNOWN] Aggregate SUM(WEIGHT) OVER () consumes feature WEIGHT whose summarizability contract is unknown because POW() has no V1 transfer rule. The compiler cannot prove SUM() preserves meaning. Recompute from a known source, or acknowledge this warning. (acknowledge with ACK-ZO7O)


,CAMPAIGN,ATTRIBUTED
0,Blog,9.662152
1,Display,26.008585
2,Email,73.663638
3,Search,40.665625


Email receives the most credit, about **73.66**, because it is closest to both purchases. Blog receives the least, about **9.66**, because it is farthest from User 1's purchase. The credits still add up to **150**.

The query returns a warning: the compiler cannot tell whether a sum of `POW()` results keeps a business meaning. Here the sum only normalizes the weights within one journey, so the warning is expected.

## Apply channel-quality weights

Keep the 30-day lookback, but weight for a different reason: an impression gets `0.3`, every other touch `1.0`. Each touch's credit is proportional to its weight:


In [14]:
%%featureql --client client

DEFINE
    credits := qualifying_touches.CARRY(conversion_ts AS conversion_ts).CARRY(
        conversion_revenue AS revenue
    ).TRANSFORM(
        DEFINE
            weight := IF(touchpoint_type = 'ad_impression', 0.3e0, 1e0)
        RETURN
            campaign,
            credit := revenue::DOUBLE * weight / SUM(weight) OVER ()
    ),
    credit := UNNEST(credits)
RETURN
    campaign := credit[campaign],
    attributed := UNSAFE SUM(credit[credit]) GROUP BY (credit[campaign])
FROM FM.MKT
FOR
    conversion_id := BIND_VALUES(ARRAY(1, 2)),
    NESTED user_id := BIND_VALUES(ARRAY(1, 2)),
    lookback_days := BIND_VALUE(30)
ORDER BY campaign
;

,CAMPAIGN,ATTRIBUTED
0,Blog,27.272727
1,Display,8.181818
2,Email,57.272727
3,Search,57.272727


Display falls to about **8.18**, compared with **22.5** under linear attribution. Blog receives about **27.27**, while Search and Email each rise to about **57.27**. The impression still participates; it carries less evidence than a click.

## What's next

You held the journeys constant while changing eligibility, selection, and weighting rules. Because the qualifying set is persisted once, each rule is a few lines on top of it, and every rule reads the same touches.

Continue with [Healthcare episodes](https://featuremesh.com/docs/tutorials/analytics/healthcare) for interval and sequence analysis, or [OBT modeling](https://featuremesh.com/docs/tutorials/analytics/obt) for deeper work inside nested arrays.


---

Source tutorial: [/docs/tutorials/analytics/marketing](https://featuremesh.com/docs/tutorials/analytics/marketing)
